# 트랜스포머 이해와 LLM 효율적 파인튜닝(PEFT) 실습
## 1일차 — Transformer 구조에서 생성형 LLM까지 (맥 버전)

## 과정 소개와 학습 흐름

### LLM과 Transformer

GPT, Llama, Gemma 등의 생성형 LLM은 **Transformer 구조를 바탕으로 만들어졌습니다.** Transformer를 배우는 것은 LLM의 내부 구조를 이해하는 과정입니다.

구조를 이해하면 모델이 문맥을 반영하고 문장을 생성하는 방식을 설명할 수 있습니다. 

파인튜닝에서 어떤 부분을 학습하는지, 모델 전체를 학습하지 않고도 동작을 조정할 수 있는지도 이 구조와 연결됩니다.

Transformer가 등장한 이유부터 살펴보면 각 구성 요소가 어떤 문제를 해결하는지 이해하기 쉽습니다. 이를 위해 먼저 Transformer 이전의 문장 처리 방식과 그 한계를 살펴봅니다.

## 학습 목표

- 문장의 순서와 문맥이 모델의 답변에 왜 중요한지 설명할 수 있다.
- Transformer가 등장하기 전의 문장 처리 방식과 그 한계를 이해한다.
- 텍스트가 숫자로 바뀌어 문맥을 반영하고 출력으로 이어지는 과정을 따라갈 수 있다.
- Transformer의 원리가 생성형 LLM에 어떻게 이어졌는지 설명할 수 있다.
- 모델을 사용해 답변을 만드는 것과 데이터로 모델을 학습하는 것을 구분할 수 있다.
- 이미 학습된 모델을 업무에 맞게 활용하고 추가 학습하는 방법을 구분할 수 있다.

## 0. 들어가기 전 준비: 벡터·행렬·행렬곱

이 절은 뒤에서 나올 Transformer 계산을 따라가기 위한 최소한의 수학입니다.  
식을 외우기보다 **숫자가 어떤 모양(shape)으로 놓여 있는지**를 보는 데 집중합니다.

### 0.1 벡터(Vector)

**벡터**는 숫자를 일렬로 늘어놓은 것입니다. 모델은 단어 하나를 숫자 여러 개로 표현하는데, 이 숫자 묶음이 벡터입니다.  
숫자의 개수가 벡터의 **차원**이고, 이 과정에서는 `d_model`이라고 부릅니다.

In [ ]:
import numpy as np  # 수치 계산 라이브러리 NumPy를 불러옵니다.

student_vec = np.array([0.2, -1.0, 0.5, 0.0, 1.3, -0.4])  # 단어 'student'를 숫자 6개로 표현합니다. 벡터의 차원은 6입니다.
print("벡터:", student_vec)
print("shape:", student_vec.shape, "→ 차원 6짜리 벡터 1개")  # shape가 (6,)이면 숫자 6개가 한 줄로 놓인 1차원 배열입니다.

### 0.2 행렬(Matrix)

**행렬**은 숫자를 행(row)과 열(column)로 정렬한 표입니다. 벡터를 여러 개 쌓으면 행렬이 됩니다.  
문장 하나는 **단어 개수 × 차원** 크기의 행렬로 표현됩니다. 행렬의 shape는 `(행 수, 열 수)`로 씁니다.

In [ ]:
sentence_mat = np.arange(12).reshape(3, 4)  # 단어 3개를 각각 숫자 4개로 표현한다고 가정하고 3행 4열 배열로 나타냅니다.
print(sentence_mat)
print("shape:", sentence_mat.shape, "→ (단어 수, 차원)")

### 0.3 행렬곱(Matrix Multiplication)

행렬곱은 **앞 행렬의 행**과 **뒤 행렬의 열**을 하나씩 짝지어 곱하고 더하는 연산입니다.  
예를 들어 `[1, 2, 3]`과 `[1, 0, 1]`의 곱은 `1×1 + 2×0 + 3×1 = 4`입니다.

$$
(m \times n) \cdot (n \times p) = (m \times p)
$$

- 가운데 숫자($n$)가 **같아야** 곱할 수 있습니다.
- 결과의 shape는 **바깥쪽 두 숫자**($m$, $p$)를 따릅니다.

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])            # A는 2행 3열 행렬입니다.
B = np.array([[1, 0],
              [0, 1],
              [1, 1]])               # B는 3행 2열 행렬입니다.

C = np.matmul(A, B)                  # 안쪽 크기 3이 같으므로 곱할 수 있고, 결과는 2행 2열입니다.
print("A @ B =\n", C)
print("결과 shape:", C.shape, "→ (2, 2)")

try:  # 안쪽 크기가 다른 행렬을 곱하면 어떤 오류가 나는지 확인합니다.
    np.matmul(A, A)                  # A @ A는 안쪽 크기 3과 2가 달라 계산할 수 없습니다.
except ValueError as e:
    print("오류:", e)

#### 내적(Dot Product)

같은 길이의 두 벡터에서 대응하는 숫자끼리 곱해 모두 더한 값입니다.  
값이 클수록 두 벡터가 **같은 방향을 향한다**, 즉 비슷하다고 볼 수 있습니다.  
Attention에서 Query와 Key를 곱해 "얼마나 관련 있는지" 점수를 매기는 것이 바로 이 내적입니다.

In [ ]:
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])
print("u · v =", np.dot(u, v), "→ 1×4 + 2×5 + 3×6 = 32")
print("u와 반대 방향 벡터(-u)의 내적 =", np.dot(u, -u), "→ 음수가 나옵니다")

### 0.4 Softmax: 점수를 확률로 바꾸기

신경망의 마지막 출력은 보통 **점수(logit)** 입니다. 점수는 음수일 수도 있고 합이 1이 아니어서 그대로는 확률이라고 할 수 없습니다.  
**Softmax**는 점수를 다음 세 단계로 바꿉니다.

1. 각 점수를 $e^{z}$로 바꿉니다. 결과는 항상 양수이고, 큰 점수는 더 크게 커집니다.
2. 전체 합으로 나눕니다.
3. 결과는 합이 1인 확률이 됩니다.

$$
\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

<img src="image/softmax_steps.svg" width="1000">

뒤에서 Attention 점수를 비율로 바꿀 때도 같은 softmax를 씁니다.

In [ ]:
def softmax(z):  # 마지막 축(행마다) 기준으로 확률을 만듭니다.
    e = np.exp(z - z.max(axis=-1, keepdims=True))  # 큰 값에서 넘치지 않도록 최댓값을 빼 줍니다.
    return e / e.sum(axis=-1, keepdims=True)  # 합으로 나누면 합이 1인 확률이 됩니다.

z = np.array([2.0, 1.0, 0.1])  # 클래스 3개에 대한 점수입니다.
p = softmax(z)  # 세 점수를 각 클래스의 확률로 바꿉니다.
print("점수 z     :", z)
print("e^z        :", np.round(np.exp(z), 2))
print("확률 p     :", np.round(p, 2))
print("확률의 합  :", p.sum())

### 0.5 간단한 신경망에서 shape가 바뀌는 모습

가장 단순한 **2층 신경망**으로 shape 변화를 봅니다. 설비 센서 값 3개를 받아 `정상 / 주의 / 고장`의 확률을 내는 모델이고, 샘플 4개를 한 번에 넣습니다.

<img src="image/shape_flow_mlp.svg" width="1000" alt="입력 노드 3개, 은닉 노드 5개, 출력 노드 3개와 가중치 W1(3×5), W2(5×3)의 shape 변화">

| 단계 | shape | 하는 일 |
|------|-------|---------|
| 입력 x | (4, 3) | 샘플 4개, 센서 값 3개 |
| 은닉 표현 | (4, 5) | x @ W1 (행렬곱) |
| 출력 | (4, 3) | 은닉 표현 @ W2 (행렬곱) |

신경망의 각 층은 0.3에서 본 **행렬곱**으로 shape를 바꿉니다. 샘플 수(4)는 그대로이고 특성 수만 3 → 5 → 3으로 바뀝니다.

## 1. Transformer 개요

Transformer가 등장한 배경을 알려면 이전 방식인 **RNN**과 **Seq2Seq**의 한계를 먼저 살펴봐야 합니다.

### 1.1 문장은 순서가 있는 데이터입니다

문장은 단어가 **순서대로** 이어진 데이터입니다.  
`나는 배가 너무 고프다`에서 `배`의 뜻은 주변 단어와 순서를 함께 봐야 알 수 있습니다.

<img src="image/text_time_sequence.png" width="420">

그래서 초기 신경망 언어 모델은 사람처럼 **앞에서부터 한 단어씩 읽으며 기억을 갱신하는 방식**을 택했습니다.  
이것이 **RNN(Recurrent Neural Network)** 입니다.

### 1.2 RNN: 기억(hidden state)을 들고 한 칸씩 읽기

Transformer가 등장하기 전에는 문장을 어떻게 처리했을까요? 대표적인 방법은 **앞에서 읽은 내용을 다음 조각을 읽을 때 넘겨주는 것**이었습니다.

예를 들어 `설비가 갑자기 멈췄다`를 읽는다고 해 봅시다. `설비가`를 읽은 내용은 `갑자기`를 처리할 때 반영되고, `갑자기`까지 반영된 내용은 다시 `멈췄다`를 처리할 때 반영됩니다.

이런 방식의 신경망을 **RNN(Recurrent Neural Network, 순환 신경망)** 이라고 합니다. RNN은 문장의 조각을 순서대로 읽으며 지금까지의 내용을 반영한 숫자 묶음을 다음 단계로 전달합니다. 0.1절에서 본 **벡터**가 바로 이 숫자 묶음이고, RNN이 문맥을 담아 넘기는 벡터를 **은닉 상태(hidden state)** 라고 합니다. 여기서는 이것을 **문맥 기억**이라고 부릅니다.

Transformer가 어떤 문제를 개선했는지 이해하기 위해, RNN의 문맥 전달 방식과 그 한계를 살펴봅니다.

아래 그림에서 현재 입력은 $x$, 문맥 기억은 $h$로 표시합니다.

<img src="image/rnn_same_cell_sequence.gif" width="800">

주황색 상자는 움직이지 않습니다. `설비가`, `갑자기`, `멈췄다`가 **하나의 같은 RNN 셀**에 차례로 들어갑니다.  
셀이 만든 기억 $h$는 다음 입력과 함께 다시 사용됩니다.  
즉, 문장의 위치마다 새로운 셀을 만드는 것이 아니라 **같은 가중치를 가진 셀을 반복해서 사용**합니다.

RNN은 **순서가 있는 데이터를 한 칸씩 읽고**, 앞에서 읽은 내용을 내부 기억에 반영합니다.  

설비 센서나 전력 사용량처럼 시간 순서가 있는 값, 그리고 단어처럼 순서가 중요한 텍스트에 사용합니다.

| 분야 | 입력 예 | 출력 예 |
|------|---------|---------|
| 설비 센서 | 1초 간격 진동값 `0.2 → 0.3 → 1.8` | 현재 설비의 이상 여부 |
| 전력 사용량 | 9시부터 시간별 사용량 `3.1 → 3.4 → 3.8 kWh` | 다음 시간의 예상 사용량 |
| 음성 | 짧은 구간으로 나눈 음성 신호의 연속 | 인식한 말 `"출발"` |
| 텍스트: 다음 단어 예측 | `오늘 회의는` | `오후` |
| 텍스트: 문장 분류 | `배송이 늦어서 불편해요` | `불만` |
| 텍스트: 개체명 인식 | `김민수가 서울에서 일한다` | `김민수: 인명`, `서울: 장소` |
| 텍스트: 번역 | `Please check the order.` | `주문을 확인해 주세요.` |

설비 값이든 단어든, RNN은 **지금 들어온 값**과 **앞 칸까지의 기억**을 함께 사용합니다.

현재 읽는 값 $x_t$와 앞 칸의 기억 $h_{t-1}$로 새 기억인 **hidden state** $h_t$를 계산합니다.

$
h_t = \tanh(x_t W_x + h_{t-1} W_h)
$

$h_t$를 계산하려면 $h_{t-1}$이 먼저 있어야 합니다.  
텍스트에서는 앞 단어, 센서 데이터에서는 앞 시점의 계산이 끝나야 다음 칸으로 넘어갈 수 있는 **순차 구조**입니다.  
모든 칸은 같은 가중치 $W_x$, $W_h$를 공유합니다.

아래 그림은 같은 예문을 시간 순서대로 펼쳐 보여 줍니다. 주황색은 계산 중인 단계, 파란색은 계산이 끝난 단계, 회색은 기다리는 단계입니다.

<img src="image/rnn_sequential_processing.gif" width="800">

앞 단계의 계산이 끝나야 다음 단계로 넘어갈 수 있어 입력 전체가 준비되어 있어도 첫 토큰부터 순서대로 계산해야 합니다.

**GPU는 여러 계산을 동시에 처리할 때 빠르지만, RNN은 앞 칸의 결과를 기다려야 하므로 한 문장의 단어들을 병렬로 처리하기 어렵습니다.**

학습 신호도 여러 칸을 거치며 작아지는 **기울기 소실(vanishing gradient)** 이 생겨, 멀리 떨어진 단어 사이의 관계(**장거리 의존성**)를 배우기 어렵습니다.  
LSTM과 GRU는 이를 완화했지만, 단어를 순서대로 처리하는 구조는 그대로입니다.

### 1.3 Seq2Seq: Encoder와 Decoder를 따로 두기

번역할 때는 원문을 읽는 일과 번역문을 만드는 일이 필요합니다.  
**Seq2Seq(Encoder-Decoder)** 는 서로 다른 가중치를 쓰는 RNN 두 개에 이 일을 하나씩 맡깁니다.

<img src="image/seq2seq.gif" width="640">

- **Encoder RNN**은 원문을 끝까지 읽고, 마지막 hidden state를 **context vector**로 넘깁니다.  
- **Decoder RNN**은 그 벡터와 앞에서 만든 단어를 받아 번역문을 이어 씁니다.

Encoder가 넘긴 context vector는 길이가 고정되어 있습니다.  
Decoder는 원문 전체를 이 벡터 하나로 전달받기 때문에, 긴 문장에서는 중요한 정보를 놓치기 쉽습니다.

#### 바로 확인하기

번역에서 Encoder와 Decoder는 각각 무엇을 하나요? 두 RNN은 같은 가중치를 쓰나요?

<details><summary>예시 답안</summary>

Encoder는 원문을 읽어 context vector를 Decoder에 넘깁니다. Decoder는 이 벡터와 앞에서 만든 단어를 바탕으로 번역문을 이어 갑니다. 두 RNN은 가중치를 공유하지 않습니다.

</details>

### 1.4 Attention의 등장: 필요한 단어를 그때그때 다시 보기

기존 Seq2Seq는 입력 문장을 고정 길이 벡터 하나에 담았습니다. [Bahdanau 등(2014)](https://arxiv.org/abs/1409.0473)과 [Luong 등(2015)](https://aclanthology.org/D15-1166/)은 Decoder가 출력 단어를 예측할 때마다 입력의 다른 위치를 다시 참고하는 **Attention**을 제안했습니다. Decoder는 단어를 하나 만들 때마다 다음 네 단계를 반복합니다.

1. **점수 계산**: 현재 Decoder 상태 $s_t$와 Encoder의 각 $h_i$를 내적해 관련도 점수를 구합니다.  
2. **softmax**: 점수를 합이 1인 가중치 $\alpha_i$로 바꿉니다.  
3. **가중합**: 가중치만큼 $h_i$를 섞어 **context vector $c_t$** 를 만듭니다.  
4. **concat**: $c_t$와 $s_t$를 이어 붙여 다음 단어의 확률을 계산합니다.

출력 단어가 바뀔 때마다 1~4단계를 새로 계산합니다. 그래서 출력 문장 전체가 같은 벡터 하나에만 의존하지 않습니다.

아래 그림은 `I am a student`를 `저는 학생입니다`로 번역하는 과정 전체에서 이 네 단계가 반복되는 모습입니다. Encoder가 입력을 한 단어씩 읽어 $h_1 \dots h_4$를 만든 뒤, Decoder가 `저는 학생입니다`를 한 단어씩 만들면서 매 순간 어떤 입력 단어에 가장 집중하는지(굵은 선·보라색 강조, %)가 바뀌는 것을 확인하세요.

<img src="image/seq2seq_attention_motion.gif" width="880">

지금까지의 변화를 한 장으로 정리하면 다음과 같습니다. Attention 전까지는 여전히 RNN으로 계산하고, Transformer는 순환 없이 토큰 관계를 직접 계산합니다.

<img src="image/seq2seq_to_transformer.svg" width="1000">


### 연결해서 확인하기

RNN 기반 Seq2Seq에 Attention을 추가했는데도 앞 계산을 기다리는 이유는 무엇인가요?

<details><summary>예시 답안</summary>

Attention은 출력이 필요한 입력 부분을 다시 참고하게 합니다. 문맥 기억을 만드는 Encoder와 Decoder는 여전히 RNN이므로 이전 상태를 기다려야 합니다.

</details>

### 1.5 그래도 남은 한계

Attention 덕분에 병목은 줄었지만, Encoder와 Decoder는 여전히 RNN이었습니다.

| 한계 | 원인 | 결과 |
|------|------|------|
| 순차 계산 | $h_t$가 $h_{t-1}$에 의존 | GPU 병렬화가 어려워 학습이 느림 |
| 장거리 의존성 | 정보가 여러 단계를 거쳐 전달됨 | 긴 문장에서 앞부분 정보가 약해짐 |
| 기울기 소실 | 같은 연산이 반복적으로 곱해짐 | 긴 시퀀스 학습이 불안정 |

앞 칸의 정보는 여러 칸을 거치며 점점 약해지므로, 멀리 떨어진 단어 사이의 관계를 기억하기 어렵습니다.

그래서 연구자들은 질문을 바꿨습니다.

> **Attention이 이렇게 유용하다면, RNN을 아예 빼고 Attention만으로 문장을 처리할 수는 없을까?**

<img src="image/rnn_vs_transformer.svg" width="860">

### 1.6 Transformer의 등장: "Attention Is All You Need" (2017)

2017년 Google 연구진은 논문 [Attention Is All You Need](https://arxiv.org/abs/1706.03762)에서 RNN 없이 **Attention만으로 만든 Transformer**를 발표했습니다.

- 모든 단어 쌍의 관계를 **행렬곱 한 번**으로 계산하므로 병렬 처리가 가능합니다.  
- 어떤 두 단어든 **한 단계 만에** 직접 연결되므로 장거리 의존성을 잘 잡습니다.  
- 그 결과 **더 적은 학습 비용으로 더 높은 번역 품질**을 달성했습니다.

<img src="image/transformer_table.png" width="700">

위 표에서 Transformer(big)는 당시 최고 수준 모델보다 BLEU(번역 품질 점수)가 높으면서도, 학습 연산량(FLOPs)은 훨씬 적습니다.

### 1.7 Transformer 전체 구조 한눈에 보기

<img src="image/transformer_architecture_annotated.svg" width="860">

원 논문의 구조도에 입력부터 출력까지의 순서를 번호로 표시했습니다. ① 입력 문장 → ② 입력 표현 → ③ Encoder → ④ 이전 출력 토큰 → ⑤ Decoder → ⑥ 출력 확률 순서로 아래에서 위로 따라가며 각 구성 요소를 살펴봅니다.  
왼쪽이 **Encoder**, 오른쪽이 **Decoder**입니다.

## 2. Transformer 입력 표현

Transformer는 글자를 직접 읽지 못합니다. 문장은 아래 과정을 거쳐 **숫자 벡터의 행렬**이 된 뒤에야 모델에 들어갑니다.

```text
문장 "I am a student."
-> 토큰화(Tokenization): ["I", " am", " a", " student", "."]
-> 정수 ID 변환:        [40, 1079, 264, 5458, 13]  (Qwen2.5 토크나이저 기준)
-> 임베딩(Embedding):   각 ID를 d_model 차원 벡터로 변환 -> (문장 길이, d_model) 행렬
-> 위치 정보 더하기:     각 위치의 순서 정보를 벡터에 더함
-> Encoder / Decoder 블록으로 입력
```

<img src="image/text_input_horizontal.png" width="560">

이 절에서 다루는 **입력 준비** 부분을 원 논문 구조도에서 강조하면 다음과 같습니다.

<img src="image/transformer_part_input.svg" width="1000">

### 2.1 토큰화(Tokenization)

토큰화는 문장을 모델이 다루는 기본 단위인 **토큰(token)** 으로 나누는 과정입니다.  
나눌 수 있는 토큰들을 모아 **사전(vocabulary)** 을 만들고, 사전에 오른 토큰마다 고유한 번호(인덱스)를 매깁니다.  
문장을 토큰화한 뒤에는 각 토큰을 이 사전에서 찾아 **번호로 치환**하는데, 이 번호가 바로 토큰 ID입니다.

<img src="image/tokenization.png" width="620">

| 방식 | 예: `unbelievable` | 장점 | 단점 |
|------|------------------|------|------|
| 단어 단위 | `unbelievable` | 사람이 이해하기 쉬움 | 사전에 없는 단어(OOV)를 처리 못함, 어휘 수가 폭발 |
| 문자 단위 | `u n b e l i e ...` | OOV 없음 | 시퀀스가 너무 길어지고 의미 단위가 약함 |
| **서브워드 단위** | `un` + `believ` + `able` | 자주 쓰는 단어는 통째로, 드문 단어는 조각으로 | 조각이 사람 눈에 어색할 수 있음 |

현대 LLM은 모두 **서브워드 토큰화**를 사용합니다. 대표 알고리즘은 다음과 같습니다.

| 알고리즘 | 핵심 아이디어 | 사용 모델 |
|----------|---------------|-----------|
| BPE (Byte Pair Encoding) | 가장 자주 붙어 나오는 문자 쌍을 반복해서 병합 | GPT 계열, Llama 3, Qwen (Byte-level BPE) |
| WordPiece | 병합했을 때 말뭉치 가능도가 가장 커지는 쌍을 병합 | BERT |
| SentencePiece (BPE/Unigram) | 공백까지 문자로 취급해 언어에 상관없이 원문에서 바로 학습 | T5, Llama 2 |

토크나이저는 **모델과 한 세트**입니다. 모델을 학습할 때 쓴 토크나이저를 그대로 써야 하며, 다른 모델의 토크나이저를 섞어 쓰면 전혀 다른 ID가 들어가 모델이 동작하지 않습니다.

In [ ]:
# BERT의 WordPiece 방식으로 영어 문장이 어떻게 나뉘는지 확인합니다.
from transformers import AutoTokenizer  # 사전학습 토크나이저를 불러옵니다.

# 참고 : https://huggingface.co/google-bert/bert-base-uncased
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")  # BERT 토크나이저를 내려받습니다.
english = "Tokenization makes unbelievably rare words manageable."

print("토큰:", bert_tokenizer.tokenize(english))  # '##'은 앞 조각에 이어 붙는 조각이라는 표시입니다.
print("ID  :", bert_tokenizer(english)["input_ids"])  # [CLS]=101, [SEP]=102 같은 특수 토큰이 앞뒤에 붙습니다.

In [ ]:
# Qwen 토크나이저가 한국어 문장을 어떻게 나누는지 확인합니다.
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")  # Qwen 토크나이저를 불러옵니다.
korean = "트랜스포머는 문장을 토큰으로 나눠서 처리합니다."
korean_ids = qwen_tokenizer(korean, add_special_tokens=False)["input_ids"]  # 특수 토큰 없이 한국어 문장을 ID로 바꿉니다.

print("토크나이저 어휘 수:", f"{len(qwen_tokenizer):,}")
print("토큰 ID:", korean_ids)
print("한 토큰씩 디코딩:", [qwen_tokenizer.decode([token_id]) for token_id in korean_ids])  # 한글 바이트가 여러 토큰으로 나뉘면 한 조각만 디코딩할 때 �가 보일 수 있습니다.
print("전체를 디코딩:", qwen_tokenizer.decode(korean_ids))  # 전체 ID를 합쳐 원문으로 되돌립니다.

결과에서 확인할 점은 다음과 같습니다.

- 토큰은 단어와 일치하지 않습니다. 한 단어가 여러 조각으로 나뉠 수 있습니다.
- 토큰 하나를 디코딩할 때 보이는 `�`는 한글의 UTF-8 바이트가 여러 토큰으로 나뉘어, 그 조각만으로는 글자가 완성되지 않았다는 뜻입니다.
- 전체 토큰 ID를 함께 디코딩하면 원래 문장으로 돌아옵니다. **토큰 ID와 사람이 읽는 글자는 서로 다른 단위**입니다.

#### 바로 확인하기

토큰과 토큰 ID는 각각 무엇인가요?

<details><summary>예시 답안</summary>

토큰은 모델이 읽는 텍스트 조각이고, 토큰 ID는 그 조각을 나타내는 정수 번호입니다.

</details>

### 2.2 토큰 수가 중요한 이유

LLM에서 토큰은 **비용과 한계의 단위**입니다.

| 항목 | 토큰과의 관계 |
|------|---------------|
| 컨텍스트 길이 | 모델이 한 번에 볼 수 있는 최대 토큰 수 (예: Qwen2.5-0.5B-Instruct는 32K 토큰) |
| API 요금 | 입력·출력 토큰 수에 비례해 과금 |
| 학습·추론 속도 | Attention 계산량이 토큰 수의 제곱에 비례 |

In [ ]:
# 같은 토크나이저로 두 문장을 나누고, 글자 수와 토큰 수를 비교합니다.
ko_text = "오늘 회의는 오후 세 시에 시작합니다."
en_text = "Today's meeting starts at three in the afternoon."
ko_ids = qwen_tokenizer(ko_text, add_special_tokens=False)["input_ids"]  # 한국어 문장의 토큰 ID입니다.
en_ids = qwen_tokenizer(en_text, add_special_tokens=False)["input_ids"]  # 영어 문장의 토큰 ID입니다.
print(f"한국어: {len(ko_text)}글자 / {len(ko_ids)}토큰")
print(f"영어  : {len(en_text)}글자 / {len(en_ids)}토큰")

글자 수가 비슷하거나 의미가 같아도 **토큰 수는 다를 수 있습니다.**
한국어 데이터의 길이를 정할 때는 글자 수 대신 실제 토크나이저로 센 토큰 수를 사용합니다.

### 2.3 임베딩(Embedding)

토큰 ID는 그저 번호일 뿐이라 `5913`번과 `5914`번이 비슷한 뜻이라는 보장이 없습니다.  
그래서 각 ID를 **의미를 담을 수 있는 실수 벡터**로 바꾸는데, 이것이 임베딩입니다.

- 임베딩 층은 `(어휘 수, d_model)` 크기의 **거대한 표**입니다.  
- 토큰 ID가 들어오면 표에서 그 번호의 행을 **그대로 꺼내옵니다(lookup)**.  
- 표의 값은 처음에는 무작위이고, 학습하면서 **비슷한 뜻의 토큰끼리 가까운 벡터**가 되도록 조정됩니다.

<img src="image/embedding_visual.svg" width="720">

#### 바로 확인하기

두 문장에 같은 토큰 ID `25`가 있습니다. 같은 모델의 임베딩 표에서 꺼내는 벡터는 같을까요? 주변 문맥을 반영한 뒤에도 항상 같을까요?

<details><summary>예시 답안</summary>

표에서 꺼내는 벡터는 같습니다. 하지만 주변에 다른 토큰이 있으면 서로 다른 정보를 반영하므로, 문맥을 처리한 뒤의 벡터는 달라질 수 있습니다.

</details>

이렇게 꺼낸 벡터가 다음 단계의 입력이 됩니다. 같은 토큰 ID는 임베딩 표에서 같은 벡터를 가져옵니다.

### 2.4 위치 정보가 필요한 이유

Self-Attention은 모든 단어를 **동시에** 봅니다. 순서대로 읽지 않으므로 그대로 두면 **단어의 순서를 구분하지 못합니다.**

<img src="image/word_order_position.svg" width="900">

왼쪽처럼 위치 표시가 없으면 두 문장은 같은 단어 카드 묶음으로 보입니다.  
오른쪽처럼 Transformer는 임베딩에 **위치 정보(Positional Encoding)** 를 더해 각 벡터에 "나는 몇 번째 단어다"라는 표시를 붙입니다.

<img src="image/positional_encoding.png" width="560">

### 2.5 사인·코사인 위치 인코딩

원 논문은 학습하지 않는 고정 함수인 **사인·코사인 위치 인코딩**을 사용했습니다.

$$
PE_{(pos,\,2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right), \qquad
PE_{(pos,\,2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)
$$

- $pos$: 단어의 위치(0, 1, 2, ...), $i$: 벡터의 차원 번호  
- 앞쪽 차원은 **빠르게 진동**하고 뒤쪽 차원은 **느리게 진동**합니다. 시계의 초침·분침·시침처럼 여러 주기를 섞어 위치마다 고유한 패턴을 만듭니다.  
- 값의 범위가 -1~1이라 임베딩에 더해도 의미 정보를 크게 해치지 않습니다.

위치 0~49의 인코딩을 512차원 벡터로 계산하면 아래와 같습니다. 행은 토큰의 위치, 열은 벡터의 차원이며 색은 값의 크기와 부호를 나타냅니다.

<img src="image/positional_encoding_heatmap.png" width="900">

첫 번째부터 세 번째 토큰에는 각각 위치 0, 1, 2의 벡터를 더합니다. 아래 임베딩 값은 덧셈을 보여주기 위한 예시이며, 세 그림은 같은 색 눈금을 사용합니다.

<img src="image/positional_encoding_addition.png" width="900">

위치 인코딩을 더하면 같은 단어라도 **몇 번째 자리에 놓였는지**가 함께 전달됩니다. 그래서 단어 순서를 바꾸면 Transformer의 입력 표현도 달라집니다.

#### 바로 확인하기

모든 위치에 같은 위치 벡터를 더하면 문장 순서를 구분할 수 있을까요?

<details><summary>예시 답안</summary>

구분하기 어렵습니다. 위치마다 다른 정보가 들어가야 같은 토큰이라도 첫 번째 위치와 다섯 번째 위치를 다르게 표현할 수 있습니다.

</details>

## 3. Attention과 Encoder

이제 Transformer의 핵심인 **Self-Attention**이 문장 속 관계를 어떻게 읽는지 그림으로 살펴봅니다.  
Self-Attention은 **한 문장 안의 단어들이 서로를 참고해 자기 표현을 문맥에 맞게 고치는 연산**입니다.

이 절에서 다루는 **Encoder** 부분을 원 논문 구조도에서 강조하면 다음과 같습니다.

<img src="image/transformer_part_encoder.svg" width="1000">

### 3.1 Self-Attention의 직관

> The monkey ate that banana because **it** was too hungry.

여기서 `it`이 가리키는 것은 `banana`가 아니라 `monkey`입니다. 사람은 `hungry`라는 단어를 보고 바로 알아챕니다.  
Self-Attention은 `it`의 벡터를 만들 때 문장의 모든 단어를 살펴보고, **관련 있는 단어(`monkey`)의 정보를 더 많이 섞어** `it`의 표현을 문맥에 맞게 바꿉니다.

<img src="image/selfattention.png" width="330">

그 결과 같은 단어라도 문장마다 벡터가 달라집니다. 이를 **문맥 임베딩(contextual embedding)** 이라고 합니다.  
`배를 타다`의 `배`와 `배가 고프다`의 `배`가 서로 다른 벡터가 되는 이유입니다.

#### 바로 확인하기

`작업자가 장비를 점검한 뒤 그것을 정지했다`에서 `그것`이 의미를 만들 때 가장 강하게 참고해야 할 토큰은 무엇일까요?

<details><summary>예시 답안</summary>

문맥상 `장비`를 강하게 참고해야 합니다. Self-Attention은 이런 토큰 사이의 관련성을 숫자로 계산합니다.

</details>

### 3.2 Query, Key, Value

Self-Attention은 각 단어 벡터를 세 가지 역할의 벡터로 바꿔서 계산합니다.  
같은 단어가 세 역할을 모두 맡는다는 점이 중요합니다. 3.1절의 예문에서 `it`을 따라가며 역할을 봅니다.

> The monkey ate that banana because **it** was too hungry.

| 벡터 | 한 줄 정의 | `it`과 함께 보면 |
|------|-----------|-----------------|
| **Query (Q)** | 이 단어가 문장에서 **무엇을 찾고 있는지** | `it`: "내가 가리키는 명사는 어디에 있지?" |
| **Key (K)** | 이 단어가 **어떤 정보를 가졌는지** 알려 주는 이름표 | `monkey`: "나는 동물인 명사이고, 행동의 주체다" |
| **Value (V)** | 이 단어가 다른 단어에게 **실제로 넘겨 줄 내용** | `monkey`: "monkey의 의미 정보" |

계산은 세 단계로 이루어집니다.

1. `it`의 Q와 각 단어의 K를 내적해서 **단어 쌍마다 점수**를 냅니다. 학습이 잘 되면 `monkey`의 K와 점수가 높고, `banana`의 K와는 낮게 나옵니다.
2. 점수를 softmax로 바꿔 합이 1인 **참고 비율**로 만듭니다.
3. 그 비율대로 각 단어의 V를 섞어서 `it`의 새 벡터를 만듭니다. `monkey`의 V가 가장 많이 섞이므로 `it`의 벡터에는 "monkey" 정보가 많이 담깁니다.

<img src="image/key-query-value.png" width="760">

같은 입력 $X$에 서로 다른 가중치 행렬 $W_Q, W_K, W_V$를 곱해 Q, K, V를 만듭니다. 이 세 행렬이 학습 대상입니다.

$$
Q = XW_Q, \qquad K = XW_K, \qquad V = XW_V
$$


### 3.3 Scaled Dot-Product Attention

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

<img src="image/mha_img_original.png" width="560">

| 단계 | 하는 일 | 의미 |
|------|---------|------|
| 1 | Q와 K를 비교 | 모든 단어 쌍이 서로 얼마나 관련 있는지 점수를 매김 |
| 2 | 점수 조정 | 특정 단어 하나에만 지나치게 쏠리지 않게 함 |
| 3 | Mask 적용 | 보면 안 되는 위치를 가림 (4장) |
| 4 | softmax | 점수를 ‘얼마나 참고할지’의 비율로 바꿈 |
| 5 | V를 섞음 | 참고 비율만큼 정보를 섞어 새 표현을 만듦 |

수식의 각 단계는 위 그림처럼 **비교하기 → 중요한 곳 고르기 → 정보 섞기**로 이해하면 됩니다.

#### 행렬 크기로 흐름 따라가기

**shape**는 숫자 배열의 크기입니다. `(10, 512)`는 10개 토큰을 각각 512개의 숫자로 표현한다는 뜻입니다. 아래 그림은 원 논문의 base 모델 설정인 벡터 차원 512, head 8개를 사용합니다. 각 head의 Q, K, V 차원은 64이며, 토큰 수 10개는 설명을 위한 예시입니다.

<img src="image/qkv_projection.svg" width="1100">

그림에서는 Q가 찾는 기준, K가 각 단어의 표지, V가 가져올 내용입니다. Q와 K를 비교해 관련 단어를 고르고, 그 단어들의 V를 섞어 새 표현을 만듭니다.

각 행은 한 단어가 문장 전체를 얼마나씩 참고했는지를 나타내며 합이 1입니다.  
지금은 학습되지 않은 무작위 가중치라 패턴에 의미가 없지만, **학습을 거치면 문법적·의미적으로 관련 있는 단어에 가중치가 모이도록** $W_Q, W_K, W_V$가 조정됩니다.


학습된 Attention head 중에는 **대명사가 가리키는 대상**을 주로 참고하는 경우도 있습니다.  
원 논문에도 비슷한 시각화가 실려 있습니다. `making`이라는 단어가 멀리 떨어진 `more difficult`에 attention을 주며 "making ... more difficult"라는 구문을 완성하는 모습입니다.

<img src="image/Attention Visualizations.png" width="620">

### 3.4 왜 $\sqrt{d_k}$로 나눌까?

벡터 차원 $d_k$가 커지면 내적 값의 크기도 함께 커집니다.  
점수가 너무 크면 softmax가 **거의 한 단어에만 1, 나머지는 0**을 주는 극단적인 분포가 되고, 이때 그래디언트가 거의 0이 되어 학습이 잘 되지 않습니다.  
$\sqrt{d_k}$로 나누면 차원과 상관없이 점수의 분산이 약 1로 유지됩니다.

스케일링은 차원이 커져도 점수의 크기가 지나치게 커지지 않도록 조절합니다. 어느 단어에 집중할지는 학습된 Q와 K가 결정합니다.

### 3.5 Attention 흐름 다시 보기

Q가 찾을 정보를 정하고, K와 비교해 참고할 위치를 고른 뒤, 그 위치의 V를 섞습니다. 이 과정을 여러 관점에서 동시에 수행하는 것이 Multi-Head Attention입니다.

#### 바로 확인하기

참고 비중이 `장비 0.7 / 작업자 0.2 / 점검했다 0.1`이라면, 문맥 벡터에는 `장비`의 V만 들어갈까요?

<details><summary>예시 답안</summary>

세 토큰의 V가 모두 들어갑니다. 장비의 정보가 가장 큰 비중을 차지하지만, 다른 두 토큰의 정보도 각각 0.2와 0.1의 비중으로 반영됩니다.

</details>

### 3.6 Multi-Head Attention

Attention을 한 번만 계산하면 한 가지 관점의 관계만 잡습니다.  
**Multi-Head Attention**은 $d_{model}$ 차원을 $h$개의 head로 나눠 **여러 관점의 Attention을 동시에** 계산한 뒤 이어 붙입니다.

<img src="image/multiheadattention.png" width="640">

- 어떤 head는 주어-동사 관계를, 어떤 head는 대명사-지시 대상을, 어떤 head는 바로 옆 단어를 주로 봅니다.  
- head마다 차원을 $d_{model}/h$로 줄이므로 전체 계산량은 head가 하나일 때와 비슷합니다.  
- 각 head의 결과를 이어 붙인 뒤(concat) 출력 행렬 $W_O$로 다시 섞습니다.

<img src="image/concat.png" width="380">


| 모델 | $d_{model}$ | head 수 $h$ | head 차원 |
|------|------------|------------|-----------|
| Transformer base (2017) | 512 | 8 | 64 |
| BERT-base | 768 | 12 | 64 |
| Qwen2.5-0.5B-Instruct | 896 | 14 | 64 |

### 3.7 Encoder 블록 구성

Encoder 블록은 **Multi-Head Attention**과 **Feed Forward Network(FFN)** 두 부분으로 이루어지고, 각 부분 뒤에 **Add & Norm**이 붙습니다.

<img src="image/tf_encoder.jpg" width="200">

<img src="image/add_norm_ffn.svg" width="880">

| 구성 요소 | 하는 일 |
|-----------|---------|
| Multi-Head Self-Attention | 단어들이 서로의 정보를 주고받음 (단어 **사이**의 연산) |
| Add (잔차 연결) | 입력을 출력에 더해 원래 정보를 보존하고, 깊게 쌓아도 그래디언트가 잘 흐르게 함 |
| Norm (Layer Normalization) | 각 벡터의 값 분포를 정규화해 학습을 안정화 |
| Feed Forward Network | 각 단어 벡터를 **개별적으로** 가공 (보통 $d_{model} \to 4d_{model} \to d_{model}$) |

Attention이 "다른 단어에서 정보를 모으는 단계"라면, FFN은 "모은 정보를 각 단어 안에서 소화하는 단계"입니다.  
LLM 파라미터의 약 2/3가 FFN에 있으며, 사실 지식의 상당 부분이 FFN에 저장된다고 알려져 있습니다.

```text
x -> Multi-Head Attention -> (x + 결과) -> LayerNorm -> FFN -> (입력 + 결과) -> LayerNorm -> 다음 블록
```

이 블록을 같은 모양으로 반복해서 쌓으면, 단어들은 매 층에서 다른 단어의 정보를 더 참고하며 표현을 다듬습니다.

**참고: Post-LN과 Pre-LN**  
원 논문은 잔차를 더한 **뒤** 정규화(Post-LN)했지만, 최신 LLM은 대부분 Attention·FFN에 들어가기 **전에** 정규화(Pre-LN)합니다.  
층을 수십 개 쌓아도 학습이 안정적이기 때문입니다. 5장에서 Qwen2.5가 어떤 방식을 쓰는지 확인합니다.

#### 바로 확인하기

`Add & Norm`은 새로운 토큰 사이의 참고 비중을 계산하는 부분일까요? 앞의 Attention과 역할이 어떻게 다를까요?

<details><summary>예시 답안</summary>

참고 비중을 계산하는 것은 Attention입니다. Add & Norm은 변환 전 입력을 더하고 벡터의 숫자 크기를 조정해, 여러 층에서 정보가 전달되고 학습이 안정적으로 이루어지도록 돕습니다.

</details>

### 연결해서 확인하기

Encoder의 Self-Attention, FFN, Add & Norm은 각각 무엇을 하나요?

<details><summary>예시 답안</summary>

Self-Attention은 토큰 사이에서 정보를 모읍니다. FFN은 그렇게 만든 벡터를 토큰별로 다시 가공합니다. Add & Norm은 입력을 더하고 숫자 크기를 조정해 여러 층의 계산과 학습을 돕습니다.

</details>

### 3.8 Encoder만 쌓으면 어디까지 할 수 있을까

Encoder 블록만 여러 층 쌓으면, 모든 단어가 **앞뒤 문맥을 모두 보는 양방향** Self-Attention으로 문장을 깊이 이해할 수 있습니다. 문장 분류, 개체명 인식, 검색용 임베딩처럼 **문장을 이해하는 작업**에 강합니다.

반면 다음 단어를 하나씩 만들어내는 **생성 작업에는 맞지 않습니다.** 생성을 하려면 "아직 만들지 않은 미래 단어를 보지 않는" 구조가 필요합니다. 그것이 다음 장의 Decoder입니다.

## 4. Transformer Decoder

Decoder는 **출력 문장을 한 토큰씩 만들어내는** 부분입니다.  
Encoder 블록과 비슷하지만 두 가지가 다릅니다.

1. **Masked Self-Attention**: 아직 생성하지 않은 미래 토큰을 보지 못하게 가립니다.  
2. **Cross-Attention**: Encoder가 만든 입력 문장의 표현을 참고합니다.

<img src="image/transformer_attention.png" width="760">

이 절에서 다루는 **Decoder** 부분을 원 논문 구조도에서 강조하면 다음과 같습니다. Masked Self-Attention과 Cross-Attention이 여기에 포함됩니다.

<img src="image/transformer_part_decoder.svg" width="1000">

### 4.1 Decoder는 무엇을 학습하나: Teacher Forcing

번역 예시 `I am a student` → `저는 학생입니다`로 Decoder의 입력과 정답을 정리하면 다음과 같습니다.  
`<시작>`은 생성을 시작하라는 **시작 토큰**, `<끝>`은 문장이 끝났다는 **종료 토큰**입니다. (논문과 코드에서는 보통 `<sos>`, `<eos>`로 씁니다.)

| 위치 | Decoder 입력 (shifted right) | 맞혀야 할 정답 |
|------|------------------------------|----------------|
| 1 | `<시작>` | `저는` |
| 2 | `<시작> 저는` | `학생` |
| 3 | `<시작> 저는 학생` | `입니다` |
| 4 | `<시작> 저는 학생 입니다` | `<끝>` |

그림의 **Outputs (shifted right)** 가 이 뜻입니다. 정답 문장을 한 칸 오른쪽으로 밀어 입력으로 넣고, 각 위치에서 **다음 토큰**을 맞히게 합니다.  
학습할 때는 모델이 앞에서 틀리게 예측했더라도 **정답 토큰을 입력으로 넣어주는데**, 이를 **Teacher Forcing**이라고 합니다.

Teacher Forcing 덕분에 4개 위치의 예측을 **한 번에 병렬로** 학습할 수 있습니다.  
단, 조건이 하나 있습니다. 위치 2에서 `학생`을 맞혀야 하는데 입력에 `학생`이 이미 보이면 안 됩니다. 그래서 마스크가 필요합니다.

### 4.2 Masked Self-Attention

**Causal Mask**는 각 위치가 **자기 자신과 그 이전 위치만** 볼 수 있도록 Attention 점수의 위쪽 삼각형을 $-\infty$로 가립니다.  
softmax를 거치면 $-\infty$ 위치의 가중치는 0이 됩니다.

<img src="image/casual_mask.png" width="560">

마스크 덕분에 각 위치는 **미래 정보 없이** 다음 토큰을 예측하도록 학습됩니다.  
이 구조를 **자기회귀(auto-regressive)** 방식이라고 하며, GPT를 포함한 모든 생성형 LLM의 기본 원리입니다.

### 4.3 Cross-Attention: Decoder가 입력 문장을 참고하는 방법

Decoder 블록의 두 번째 Attention은 **Query는 Decoder에서, Key와 Value는 Encoder 출력에서** 가져옵니다.

<img src="image/cross_attention_flow.svg" width="820">

- Decoder: "지금 `학생` 다음 단어를 만들려는데, 입력 문장 중 어디를 봐야 하지?" -> **Query**  
- Encoder 출력: 입력 문장 `I am a student` 각 단어의 문맥 표현 -> **Key, Value**

1장에서 본 Seq2Seq + Attention의 역할을 Transformer에서는 Cross-Attention이 맡습니다.  
Query와 Key의 길이가 달라도 되므로 가중치 행렬은 `(출력 길이, 입력 길이)` 모양이 됩니다.

### 4.4 Decoder 블록과 전체 Transformer 조립

Decoder 블록은 세 개의 층으로 이루어지며, 각 층 뒤에 **Add & Norm**이 붙습니다.

<img src="image/decoder_block_flow.svg" width="760">

전체 구조(1.7절 그림)에서 Encoder가 입력 문장을 읽고, Decoder가 그 결과를 참고해 출력 문장을 한 단어씩 만듭니다.

Decoder를 맨 위까지 쌓고 나면, 마지막 블록이 내놓은 벡터를 실제 **다음 토큰**으로 바꾸는 과정이 한 번 더 필요합니다. **Linear** 층이 그 벡터를 사전(2.1절)의 토큰 수만큼의 점수(로짓)로 바꾸고, **Softmax**가 점수를 합이 1인 확률분포로 바꿉니다.

<img src="image/linear_softmax_output.svg" width="880">

이 절에서 다루는 **Linear·softmax 출력** 부분을 원 논문 구조도에서 강조하면 다음과 같습니다.

<img src="image/transformer_part_output.svg" width="1000">

예를 들어 `안전 점검 보고서를` 뒤에 올 조각을 예측한다고 가정합니다. 아래는 단어 단위로 단순화한 후보와 확률 예시입니다.

| 후보 | 예측 확률 예시 |
|---|---:|
| 요약 | 46% |
| 작성 | 22% |
| 확인 | 18% |
| 삭제 | 3% |
| 그 외 후보 전체 | 11% |

생성 설정에 따라 가장 높은 확률의 토큰을 고르거나, 이 확률을 바탕으로 무작위로 선택합니다. 선택한 토큰을 입력 뒤에 붙이고 반복하면 문장이 만들어집니다.

#### 바로 확인하기

Transformer의 다음 토큰 예측을 분류 문제로 볼 수 있는 이유는 무엇일까요?

<details><summary>예시 답안</summary>

현재 문맥을 입력으로 받아 어휘에 있는 각 토큰의 확률을 계산하고, 그중 하나를 다음 토큰으로 선택하기 때문입니다.

</details>

### 4.5 학습과 추론은 다르게 동작합니다

| 구분 | 학습(Training) | 추론(Inference) |
|------|----------------|-----------------|
| Decoder 입력 | 정답 문장을 한 칸 민 것 (Teacher Forcing) | 지금까지 **모델이 생성한** 토큰 |
| 계산 방식 | 모든 위치를 **한 번에 병렬** 계산 (마스크 사용) | 한 토큰씩 **반복** 생성 |
| 끝나는 시점 | 정답 길이만큼 | `<끝>`(종료 토큰)이 나오거나 최대 길이에 도달 |

생성할 때는 방금 만든 단어를 다음 입력으로 다시 넣어야 하므로, 학습과 달리 한 단어씩 진행됩니다.

#### 학습 계산과 답변 생성의 차이

학습할 때는 정답 문장을 알고 있으므로, 정답을 한 칸 오른쪽으로 옮겨 Decoder에 한 번에 넣을 수 있습니다. 각 위치보다 뒤의 토큰과 맺는 Attention 연결은 Causal Mask로 가립니다.

같은 층에서 앞 위치의 계산 결과를 기다리지 않고 모든 위치를 함께 계산할 수 있습니다. 각 층은 이전 층에서 나온 벡터들을 사용합니다.

반면 실제 생성에서는 아직 다음 토큰이 정해지지 않았습니다. 방금 선택한 토큰을 입력에 붙인 다음 그 뒤의 토큰을 계산하므로, 답변은 한 토큰씩 만들어집니다.

<img src="image/self_attention_parallel_mask.svg" width="1100">

#### 바로 확인하기

1. Decoder 학습에서 Causal Mask가 필요한 이유는 무엇인가요?
2. 위치별 학습 계산은 병렬로 할 수 있어도 실제 생성은 왜 한 토큰씩 진행될까요?

<details><summary>예시 답안</summary>

1. 현재 위치가 오른쪽의 미래 정답을 미리 보지 못하게 하기 위해서입니다.
2. 생성할 때는 방금 선택한 토큰이 다음 입력에 포함되어야 하므로 아직 생성하지 않은 토큰을 미리 계산할 수 없습니다.

</details>

### 4.6 작은 영한 번역기 직접 학습하기

[영한 병렬 문장 데이터셋](https://huggingface.co/datasets/strongminsu/ko-en-structured-translations)은 김민수 강사가 직접 구성해 Hugging Face Hub에 공개했습니다. **영어 `en`을 입력**, **한국어 `ko`를 정답**으로 사용합니다.

| 분할 | 문장쌍 수 | 쓰임 |
|------|----------:|------|
| train | 7,594 | 가중치 학습 |
| validation | 949 | 학습 중 손실 확인 |
| test | 950 | 학습 후 새 문장 확인 |

전체 9,493쌍 가운데 학습 데이터로 작은 Encoder-Decoder Transformer를 **처음부터(사전학습 없이) 학습**합니다. 4.1~4.5절에서 본 Teacher Forcing, Masked Self-Attention, Decoder 조립을 직접 구현하는 대신, Hugging Face의 `Seq2SeqTrainer`가 그 과정을 대신 처리하게 합니다.

In [ ]:
# 영어 문장과 한국어 번역이 짝지어진 데이터와 토크나이저를 불러옵니다.
from datasets import load_dataset  # 공개 영한 문장쌍을 불러옵니다.
from transformers import AutoTokenizer  # 영어와 한국어를 토큰 ID로 바꿉니다.

translation_data = load_dataset("strongminsu/ko-en-structured-translations")  # 학습·검증·평가용 영어·한국어 문장쌍을 불러옵니다.
translation_tokenizer = AutoTokenizer.from_pretrained("tunib/electra-ko-en-small")  # 영어와 한국어 문장을 모델이 읽을 토큰 ID로 바꿉니다.
pad_id = translation_tokenizer.pad_token_id  # 길이가 다른 문장을 배치로 묶을 때 빈자리를 채우는 토큰의 ID입니다.
bos_id = translation_tokenizer.cls_token_id  # Decoder가 번역을 시작할 때 맨 앞에 넣는 토큰의 ID입니다.
eos_id = translation_tokenizer.sep_token_id  # 번역문의 끝을 표시하는 토큰의 ID입니다.
print({name: len(rows) for name, rows in translation_data.items()})  # 학습·검증·평가 데이터에 문장쌍이 각각 몇 개 있는지 확인합니다.
print("영어:", translation_data["train"][0]["en"])
print("한국어:", translation_data["train"][0]["ko"])

#### 문장을 토큰 ID로 바꾸기

같은 토크나이저로 영어 원문과 한국어 정답을 각각 토큰화합니다. `text_target=`에 한국어를 넣으면 Decoder가 맞힐 `labels`까지 한 번에 만들어집니다.

In [ ]:
MAX_LENGTH = 32  # 영어 입력과 한국어 정답을 각각 최대 32토큰으로 제한합니다.

def tokenize_translation(batch):  # 영어는 입력, 한국어는 Decoder가 맞힐 정답으로 토큰화합니다.
    return translation_tokenizer(  # 영어 입력의 토큰 ID와 한국어 정답의 토큰 ID를 함께 만듭니다.
        batch["en"],  # Encoder가 읽을 영어 문장입니다.
        text_target=batch["ko"],  # Decoder가 맞힐 한국어 정답입니다.
        truncation=True,  # 32토큰을 넘는 부분은 자릅니다.
        max_length=MAX_LENGTH,  # 영어와 한국어에 같은 상한을 적용합니다.
    )

tokenized = translation_data.map(  # train·validation·test를 한 번에 토큰화합니다.
    tokenize_translation,  # 각 문장쌍에 위 토큰화 함수를 적용합니다.
    batched=True,  # 여러 문장쌍을 한 번에 변환합니다.
    remove_columns=translation_data["train"].column_names,  # 원본 문자열 열은 제외합니다.
)

#### Encoder-Decoder Transformer 만들기

`MarianConfig`로 작은 Transformer의 크기(층 수, head 수, 차원)를 정하고, 사전학습 가중치 없이 `MarianMTModel`을 무작위로 초기화합니다. `decoder_start_token_id`가 4.1절에서 본 "정답을 한 칸 밀어 넣을 때의 시작 토큰" 역할을 합니다.

In [ ]:
from transformers import MarianConfig, MarianMTModel  # 작은 Encoder-Decoder Transformer를 만듭니다.

translation_config = MarianConfig(  # 모델 크기와 특수 토큰을 정합니다.
    vocab_size=len(translation_tokenizer),  # 토크나이저의 전체 토큰 수입니다.
    d_model=256,  # 토큰 벡터의 크기입니다.
    encoder_layers=2, decoder_layers=2,  # Encoder·Decoder를 두 층씩 쌓습니다.
    encoder_attention_heads=4, decoder_attention_heads=4,  # Attention을 네 관점으로 나눕니다.
    encoder_ffn_dim=512, decoder_ffn_dim=512,  # FFN의 중간 벡터 크기입니다.
    max_position_embeddings=MAX_LENGTH,  # 모델이 처리할 입력과 번역문의 최대 길이를 토큰 수로 지정합니다.
    pad_token_id=pad_id,  # 문장 길이를 맞추기 위해 빈자리에 넣을 토큰의 ID입니다.
    eos_token_id=eos_id,  # 번역이 끝났음을 알리는 토큰의 ID입니다.
    decoder_start_token_id=bos_id,  # Decoder가 번역을 시작할 때 맨 앞에 넣는 토큰의 ID입니다.
)
translation_model = MarianMTModel(translation_config)  # 사전학습 가중치 없이 설정값만으로 초기화합니다.
print(f"파라미터 수: {sum(p.numel() for p in translation_model.parameters()):,}")

#### 직접 학습하기

`DataCollatorForSeq2Seq`가 배치마다 패딩과 **Decoder 입력을 한 칸 밀어 넣는 과정**(정답 -> decoder 입력)을 처리합니다. `Seq2SeqTrainer`는 2일차에서 쓸 `SFTTrainer`와 같은 계열의 Trainer로, 손실 계산·역전파·가중치 업데이트 루프를 전부 대신합니다.

In [ ]:
import platform  # 맥의 칩 종류(Apple Silicon 또는 인텔)를 확인합니다.
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer  # 문장 길이를 맞추고 번역 모델을 학습할 도구를 불러옵니다.

data_collator = DataCollatorForSeq2Seq(translation_tokenizer, model=translation_model)  # 문장 길이를 맞추고, 정답 문장을 한 칸 밀어 Decoder 입력을 만듭니다.

translation_args = Seq2SeqTrainingArguments(  # 반복 횟수, 배치 크기, 학습률과 검증 주기 등을 정합니다.
    output_dir="outputs/translation-scratch",  # 학습 결과가 저장될 폴더를 지정합니다.
    num_train_epochs=10,  # 전체 학습 데이터를 열 번 반복합니다.
    per_device_train_batch_size=32,  # 한 번의 학습 단계에서 문장쌍 32개를 처리합니다.
    per_device_eval_batch_size=32,  # 한 번의 검증 단계에서 문장쌍 32개를 처리합니다.
    learning_rate=5e-4,  # 학습할 때 가중치를 얼마나 크게 바꿀지 정하는 학습률입니다.
    eval_strategy="epoch",  # epoch마다 검증 손실을 확인합니다.
    logging_steps=50,  # 학습 50단계마다 손실 등을 기록합니다.
    report_to="none",  # 학습 기록을 외부 서비스로 보내지 않습니다.
    save_strategy="no",  # 학습 중간에는 모델을 저장하지 않습니다.
    use_cpu=platform.machine() != "arm64",  # 인텔 맥에서는 CPU로 학습합니다. 지정하지 않으면 학습 도구가 맥 GPU를 스스로 고를 수 있습니다.
)

translation_trainer = Seq2SeqTrainer(  # 번역 모델, 데이터, 학습 조건을 Trainer에 연결합니다.
    model=translation_model,  # 위에서 만든 번역 모델의 가중치를 학습합니다.
    args=translation_args,  # 앞에서 정한 배치 크기와 학습률 등의 조건을 적용합니다.
    train_dataset=tokenized["train"],  # 이 데이터로 모델의 가중치를 업데이트합니다.
    eval_dataset=tokenized["validation"],  # 이 데이터로 학습 중 번역 성능을 확인합니다.
    data_collator=data_collator,  # 길이가 다른 문장을 패딩하고 Decoder 입력을 준비합니다.
    processing_class=translation_tokenizer,  # 영어·한국어 문장을 토큰 ID로 바꾸는 토크나이저를 전달합니다.
)
translation_trainer.train()

#### 학습한 모델로 번역하기

`generate()`가 한 토큰씩 고르고 다시 입력에 붙이는 과정을 전부 처리합니다 — 5장에서 Qwen으로 할 것과 같은 방식입니다.

In [ ]:
test_example = translation_data["test"][0]  # 학습에 쓰지 않은 문장쌍입니다.
inputs = translation_tokenizer(  # 평가할 영어 문장을 모델에 넣을 토큰 ID로 바꿉니다.
    test_example["en"],  # 평가 데이터의 영어 문장을 번역 입력으로 사용합니다.
    return_tensors="pt",  # 모델에 바로 넣을 수 있도록 PyTorch 텐서로 받습니다.
    truncation=True,  # 정해 둔 길이를 넘는 문장은 뒤쪽 토큰을 자릅니다.
    max_length=MAX_LENGTH,  # 입력 문장의 길이를 최대 32토큰으로 제한합니다.
).to(translation_model.device)  # 입력과 모델이 같은 CPU 또는 GPU에 있도록 옮깁니다.
output_ids = translation_model.generate(  # 한 토큰씩 고르며 번역 문장을 만듭니다.
    input_ids=inputs["input_ids"],  # 번역할 영어 문장의 토큰 ID를 전달합니다.
    attention_mask=inputs["attention_mask"],  # 실제 토큰과 패딩을 구분하는 마스크를 전달합니다.
    max_new_tokens=MAX_LENGTH,  # 번역을 만들 때 새 토큰을 최대 32개 생성합니다.
)

print("영어 원문:", test_example["en"])
print("한국어 정답:", test_example["ko"])
print("모델 번역:", translation_tokenizer.decode(output_ids[0], skip_special_tokens=True))
print("평가 손실:", round(translation_trainer.evaluate(tokenized["test"])["eval_loss"], 3))  # 평가 손실이 낮을수록 정답 토큰에 높은 확률을 준 것입니다.

학습 손실만 줄고 검증 손실이 다시 커지면 학습 문장을 외우기 시작했을 수 있습니다. 위의 테스트 문장에서는 정답과 생성 결과를 직접 비교해 보세요. 작은 모델을 처음부터 학습한 결과라 번역이 어색할 수 있지만, **정답을 한 칸 밀어 학습하는 과정**과 **생성한 토큰을 다시 입력하는 과정**은 분명히 확인할 수 있습니다.

### 4.7 Transformer의 세 가지 변형

Transformer의 Encoder와 Decoder는 따로 떼어 쓸 수도 있습니다.

| 구조 | 사용하는 부분 | Attention 방향 | 대표 모델 | 잘하는 일 |
|------|--------------|----------------|-----------|-----------|
| Encoder-only | Encoder | 양방향 | BERT, RoBERTa | 분류, 개체명 인식, 검색 임베딩 |
| Decoder-only | Decoder (Cross-Attention 제거) | 단방향(causal) | GPT, Llama, Qwen | 텍스트 생성, 대화, 범용 LLM |
| Encoder-Decoder | 둘 다 | Encoder 양방향 + Decoder 단방향 | 원조 Transformer, T5, BART | 번역, 요약 |

<img src="image/bert_gpt.jpg" width="560">

### 4.8 Transformer 이후의 발전

Transformer는 번역 모델로 시작했지만, 곧 거의 모든 언어 모델의 기본 구조가 되었습니다.

| 연도 | 모델 | 의미 |
|------|------|------|
| 2017 | Transformer | RNN 없이 Attention만으로 번역 |
| 2018 | GPT-1 | 대규모 사전학습 후 파인튜닝하는 흐름을 제시 |
| 2018 | BERT | 양방향 문맥 이해, 분류·검색에 강함 |
| 2019 | GPT-2, T5 | 규모를 키우면 범용 능력이 생김을 보여줌 |
| 2020 | GPT-3 (175B) | 예시 몇 개만으로 새 작업 수행(in-context learning) |
| 2022 | InstructGPT, ChatGPT | 지시를 따르는 대화형 모델의 대중화 |
| 2023 | LLaMA, Llama 2, Mistral | 공개 LLM 생태계와 LoRA 파인튜닝 확산 |
| 2024~2025 | Llama 3·4, Qwen2.5·3, DeepSeek | 장문 맥락, 멀티모달, 추론 특화, 경량 모델 |

오늘날 우리가 쓰는 생성형 LLM은 대부분 **Decoder 블록만 쌓은 구조**입니다. 이 구조가 왜 유리한지는 4장에서 자세히 다룹니다.  
그래서 1일차 후반부는 이 구조를 중심으로 LLM을 살펴봅니다.

### 연결해서 확인하기

1. 고객 문의 분류, 영문 계약서 번역, 계속 이어지는 챗봇 답변에는 각각 어떤 구조가 잘 맞을까요?
2. Decoder-only가 대화와 글·코드 생성에 같은 계산 방식을 사용할 수 있는 이유는 무엇인가요?
3. LLM이 도구를 호출하라는 텍스트를 만들기만 하면 그 도구가 실제로 실행될까요?

<details><summary>예시 답안</summary>

1. 분류에는 Encoder-only, 번역에는 Encoder–Decoder, 이어지는 문장 생성에는 Decoder-only가 잘 맞습니다. 특정 작업에 대한 최종 성능은 데이터와 학습 조건도 함께 봐야 합니다.
2. 대화와 글·코드를 모두 토큰 나열로 표현하고, 앞선 문맥에서 다음 토큰을 예측하며 이어 쓸 수 있기 때문입니다.
3. 아닙니다. 요청을 해석해 실제 도구를 실행하고 결과를 돌려주는 프로그램이 필요합니다.

</details>

## 5. Decoder-only LLM: Transformer에서 GPT와 생성형 LLM으로

### 5.1 GPT: Decoder만 쌓은 언어 모델

2018년 OpenAI의 GPT는 Transformer에서 **Decoder만** 가져왔습니다. 입력 문장을 따로 인코딩할 필요가 없으므로 Cross-Attention도 뺐습니다.

<img src="image/GPT.png" width="520">

GPT가 학습하는 것은 단 하나, **다음 토큰 예측(Next Token Prediction)** 입니다.

$$
p(x_1, \dots, x_T) = \prod_{t=1}^{T} p(x_t \mid x_{<t})
$$

- 입력과 출력을 따로 나누지 않고, **하나의 긴 텍스트**를 앞에서부터 읽으며 매 위치에서 다음 토큰을 맞힙니다.  
- 인터넷의 텍스트만 있으면 정답 레이블을 사람이 만들 필요가 없습니다. **텍스트 자체가 정답**입니다(자기지도학습).  
- 번역, 요약, 질의응답도 모두 "이어질 텍스트 생성"으로 바꿔 풀 수 있습니다.

```text
"대한민국의 수도는 서울이다."
위치 1: "대한민국의"           -> 정답 " 수도는"
위치 2: "대한민국의 수도는"     -> 정답 " 서울"
위치 3: "대한민국의 수도는 서울" -> 정답 "이다"
...
```

### 5.2 사전학습: 언어와 지식을 배우는 단계

다음 토큰을 맞히는 단순한 과제지만, 잘 맞히려면 문법, 어휘, 세계 지식, 간단한 추론까지 알아야 합니다.

| 배우는 것 | 사전학습 중 만나는 문장 예시 |
|-----------|------------------------------|
| 문법 | 그는 어제 도서관에 {갔다 / 간다} |
| 세계 지식 | 대한민국의 수도는 {서울 / 부산}이다 |
| 감정 | 이 영화는 지루해서 {최악 / 최고}이었다 |
| 계산 | 5 × 6 = {30 / 42} |
| 코드 | `for i in range(10):` 다음 줄의 들여쓰기 |

사전학습은 수천 개의 GPU로 몇 주~몇 달이 걸리는 작업이라 개별 기관이 직접 하기는 어렵습니다.  
그래서 실무에서는 **공개된 사전학습 모델(Llama, Qwen 등)을 가져와 그다음 단계(2일차에서 다룰 SFT 등)부터 진행**하는 경우가 대부분입니다.

### 5.3 규모를 키우자 생긴 일: Scaling Law

GPT-1(2018, 1.17억), GPT-2(2019, 15억), GPT-3(2020, 1,750억)는 모두 같은 Decoder-only 구조를 더 크게 만든 모델입니다.

<img src="image/gpt_parameter_growth.svg" width="900">


- **Scaling Law** ([Kaplan et al., 2020](https://arxiv.org/abs/2001.08361)): 모델 크기, 데이터 양, 계산량을 늘리면 손실이 예측 가능한 비율로 꾸준히 줄어듭니다.  
- **In-context Learning**: 파인튜닝 없이 프롬프트에 예시 몇 개만 넣어도 새로운 작업을 수행합니다(few-shot).  
- **Chinchilla 법칙** ([Hoffmann et al., 2022](https://arxiv.org/abs/2203.15556)): 같은 계산 예산이라면 모델만 키우기보다 **파라미터 1개당 약 20토큰**의 데이터로 학습하는 편이 효율적입니다. 최근 소형 모델은 이보다 훨씬 많은 토큰으로 학습해 작은 크기로도 좋은 성능을 냅니다.

### 5.4 최신 LLM 구조는 무엇이 달라졌나

기본 뼈대(Decoder 블록 반복)는 같지만, 정규화·위치 인코딩·FFN 같은 세부 부품은 효율을 위해 많이 바뀌었습니다. Qwen도 세부 구현은 조금씩 다르지만, 지금까지 배운 **Decoder-only 구조의 핵심(Masked Self-Attention + FFN 반복)** 은 그대로입니다.

### 5.5 실습: Qwen2.5-0.5B-Instruct 구조 들여다보기

<img src="image/qwen_logo.png" width="100">

**Qwen**은 Alibaba Cloud가 공개한 **Decoder-only** LLM 패밀리입니다. 0.5B부터 72B까지 여러 크기로 공개되어 있고, Apache 2.0 라이선스라 상업적으로도 자유롭게 쓸 수 있습니다. 이 과정에서는 가장 작은 **Qwen2.5-0.5B-Instruct**(약 4.9억 개 파라미터)를 씁니다.

- `Instruct`는 사전학습 뒤 **지시를 따르도록 추가 학습한** 버전이라는 뜻입니다.
- CPU에서도 실행할 수 있습니다. 맥에서는 모델 구조만 확인하므로 CPU에 32비트(기본 정밀도)로 불러옵니다.
- Attention과 FFN의 선형층 이름을 확인하면 LoRA를 붙일 위치를 고를 수 있습니다.

설정값은 [공식 모델 설정](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/blob/main/config.json)에서 확인할 수 있습니다.

In [ ]:
# Qwen 모델과 토크나이저를 불러옵니다. 첫 실행에는 약 1GB를 다운로드합니다.
import torch  # 모델의 정밀도(dtype)를 지정할 때 씁니다.
from transformers import AutoModelForCausalLM, AutoTokenizer  # 생성 모델과 토크나이저를 불러옵니다.

model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # 사용할 Qwen 모델 이름입니다.

tokenizer = AutoTokenizer.from_pretrained(model_name)  # Qwen 전용 토크나이저입니다.
model = AutoModelForCausalLM.from_pretrained(  # 이미 학습된 Qwen 모델의 가중치를 불러옵니다.
    model_name,  # 위에서 지정한 Qwen 모델 이름을 전달합니다.
    dtype=torch.float32,  # 맥에서는 CPU에 32비트로 불러옵니다. 맥 GPU에 16비트로 올리면 인텔 맥 등에서 오류가 납니다.
)

In [ ]:
print(model)  # 모듈 구조를 그대로 출력합니다. 같은 층은 "(0-23)"처럼 한 번만 보여줍니다.

Cross-Attention이 없다는 점도 확인하세요. Decoder-only 모델이기 때문입니다.

Qwen은 Decoder 블록을 여러 층 쌓은 구조입니다. 단어를 서로 연결하는 Attention과 단어별 표현을 다듬는 FFN이 함께 반복됩니다.

## 6. LLM 학습과 활용

### 6.1 LLM이 만들어지는 세 단계

ChatGPT 같은 대화형 LLM은 보통 세 단계를 거쳐 만들어집니다.

<img src="image/llm_training_stages.svg" width="900">

| 단계 | 데이터 | 학습 목표 | 결과 |
|------|--------|-----------|------|
| 1. 사전학습 (Pre-training) | 웹·책·코드 등 수조 개 토큰의 일반 텍스트 | 다음 토큰 예측 | **Base 모델**: 언어와 지식을 익혔지만 대화 형식에는 익숙하지 않음 |
| 2. 지도 미세조정 (SFT) | 사람이 만든 (지시, 모범 답변) 수천~수십만 쌍 | 답변 부분의 다음 토큰 예측 | **Instruct 모델**: 지시를 따르고 대화 형식으로 답함 |
| 3. 선호도 정렬 (Preference Alignment) | 같은 질문의 (더 좋은 답, 덜 좋은 답) 쌍 | 사람이 선호하는 답의 확률을 높임 | 더 도움되고 안전한 모델 |


SFT와 선호도 정렬은 둘 다 (지시, 답변) 데이터를 쓰지만, 모델에게 가르치는 방식이 다릅니다. `환불 규정을 알려줘`라는 질문을 예로 들면 다음과 같습니다.

- **SFT**: 모범 답변 하나만 주고, 그 답을 그대로 **따라 쓰도록** 학습합니다. (`"네, 구매 후 7일 이내에 환불 가능합니다. ..."`)
- **선호도 정렬**: 같은 질문에 대한 답변 두 개를 비교해서 더 나은 쪽의 확률은 높이고 덜 나은 쪽의 확률은 낮춥니다.  
 (`더 친절하고 근거를 갖춘 답` > `퉁명스럽거나 부정확한 답`)

SFT가 "이렇게 답하라"는 정답 하나를 외우는 것이라면, 선호도 정렬은 "이 답이 저 답보다 낫다"는 **상대적인 기준**을 배우는 것입니다.

1단계 사전학습은 5장에서 GPT와 함께 이미 살펴봤습니다. 2일차에 우리가 할 일은 이 중 **2단계 SFT를 작은 규모로, 그리고 LoRA로 효율적으로** 수행하는 것입니다.

### 6.2 LLM을 업무에 맞추는 세 가지 방법: Prompt, RAG, Fine-tuning

공개 LLM이나 상용 API를 우리 업무에 맞추는 방법은 크게 세 가지입니다. **무엇을 바꾸는가**로 구분하면 쉽습니다.

<img src="image/prompt_rag_finetune.svg" width="900">

| 구분 | Prompt Engineering | RAG (검색 증강 생성) | Fine-tuning (SFT, PEFT) |
|------|--------------------|----------------------|--------------------------|
| 바꾸는 것 | 모델에 주는 **지시문** | 모델에 주는 **참고 자료(컨텍스트)** | 모델의 **가중치** |
| 준비물 | 좋은 프롬프트 | 문서 DB, 검색 시스템 | 학습 데이터, GPU |
| 비용·시간 | 가장 적음 (즉시) | 중간 (검색 시스템 구축) | 가장 큼 (데이터 구축 + 학습) |
| 최신 정보 반영 | 어려움 | **쉬움** (문서만 교체) | 어려움 (재학습 필요) |
| 근거 제시 | 어려움 | **쉬움** (검색된 문서 인용) | 어려움 |
| 잘 바뀌는 것 | 간단한 형식·역할 | 사실 정확도, 사내 지식 | **일관된 형식·말투·절차, 도메인 표현**, 짧은 프롬프트 |
| 한계 | 긴 프롬프트 비용, 복잡한 규칙은 잘 안 지킴 | 검색 품질에 좌우됨 | 새 사실 주입에 약함, 과적합, 모델 관리 부담 |

실무 판단 순서는 보통 다음과 같습니다.

```text
1. 프롬프트로 해결되는가?                           -> 예: Prompt Engineering으로 시작
2. 모델이 모르는 사내 지식·최신 정보가 필요한가?        -> 예: RAG
3. 형식·말투·절차를 매번 일관되게 지켜야 하는가?
   또는 긴 프롬프트 비용·지연 시간을 줄여야 하는가?
   또는 데이터를 외부 API로 보낼 수 없어 작은 공개 모델을 써야 하는가? -> 예: Fine-tuning (PEFT)
4. 실제 서비스는 대부분 조합: 파인튜닝한 모델 + RAG + 프롬프트
```

#### 바로 확인하기

매주 바뀌는 반품 규정을 근거와 함께 답해야 합니다. Prompt, RAG, Fine-tuning 중 무엇을 먼저 검토해야 할까요?

<details><summary>예시 답안</summary>

RAG를 먼저 검토합니다. 자주 바뀌는 지식은 모델 가중치에 넣기보다 최신 문서를 검색해 연결하는 편이 갱신과 근거 제시에 유리합니다.

</details>

## 7. 1일차 정리

| 구분 | 핵심 내용 |
|------|-----------|
| RNN·Seq2Seq의 한계 | 순차 계산으로 병렬화가 어렵고, 고정 길이 벡터 병목과 장거리 의존성 문제가 있음 |
| Attention·Transformer | 필요한 위치를 가중치로 골라 참고하며, 모든 단어 쌍을 행렬곱으로 한 번에 계산 ($\text{softmax}(QK^\top/\sqrt{d_k})V$) |
| 입력 표현 | 토큰화 -> 정수 ID -> 임베딩 + 위치 정보 |
| Encoder·Decoder 블록 | Attention과 FFN 사이에 Add & Norm. Decoder는 Masked Self-Attention과 Cross-Attention을 추가 |
| Decoder-only LLM | Cross-Attention 없이 Decoder 블록만 쌓아 다음 토큰 예측으로 학습 (GPT, Qwen) |
| 학습과 추론 | 학습은 정답을 넣고 위치별로 병렬 계산, 추론은 토큰을 하나씩 이어 생성 |
| 업무 적용 방법 | Prompt / RAG / Fine-tuning을 목적에 맞게 조합 |

```text
Transformer -> 단어 간 관계를 Attention으로 계산
Decoder-only LLM -> Causal Mask로 다음 토큰을 예측하도록 대규모 학습
2일차 -> SFT 데이터 구성, LoRA 파인튜닝, 평가
```
